# Symbolic Fall Detector v2 (URFD)

Modified version of `symbolic_fall_detector.ipynb`. Reference: Applied Sciences 2026, 16(4), 1895, Eq. 8 to 10.

Run from the project root (or set `PROJECT_ROOT`), then **Run All**. It reads our preprocessed URFD files and evaluates on the same scene-disjoint folds as the ST-GCN (`splits/split_scene_v2.json`). Thresholds are fitted on each fold's training videos only.

## 1. What changed and why
```
symbolic_v2.py
Symbolic stream, version 2, built to work together with the neural ensemble (neural_ensemble.py).

What changed from the teammate's notebook (symbolic_fall_detector.ipynb) and why
  1. Causal everywhere. The floor reference, the body scale, the gap filling and the smoothing only use
     the current and past frames, like the ST-GCN. The notebook used percentiles over the whole video,
     centred smoothing and interpolation from future frames.
  2. Hip fallback without the upward offset. When the shoulders disappear on the floor, the notebook put
     the neck 0.35 torso lengths ABOVE the hip, which raises the height exactly while the person lies down.
     We use the hip midpoint itself, as our preprocessing does.
  3. Confidence threshold 0.3, as in the paper (the notebook used 0.25).
  4. Two outputs instead of one, because the two streams fail in different ways:
       trigger  = paper Eq. 8 AND Eq. 9 (neck low AND fast descent), smoothed causally (Eq. 10).
                  High precision, low recall: used in the OR of the fusion.
       support  = the person is in a fall state: it starts when the neck gets low within 1.5 s of a fast
                  descent and stays on while the neck stays low. Used as the veto of the fusion: the neural
                  stream raises many alarms while people stand, sit or lie down slowly; those frames have
                  no fall state and are vetoed.
  5. The support height threshold is not tuned for MCC. It is fixed by a rule on the training videos only:
     the 95th percentile of the height over training frames that URFD labels "lying on the ground" (label 1),
     so the fall state covers about 95% of lying frames by construction.

Height is still a relative body height (about 1 when standing), not metres: URFD has no control points
for the paper's matrix W. It follows the Kinect floor distance closely (median correlation 0.97 on falls).

Fusion (paper Eq. 11 to 13): fused(t) = (neural(t) OR trigger(t)) AND support anywhere in the last 80 frames.
```

In [1]:
import os, json
import numpy as np
import pandas as pd

import numpy as np

FPS = 30.0
CONF = 0.3
L_SH, R_SH, L_HIP, R_HIP, L_ANK, R_ANK, NOSE = 5, 6, 11, 12, 15, 16, 0


def _mid(k, i, j):
    a, b = k[i], k[j]
    if a[2] >= CONF and b[2] >= CONF:
        return 0.5 * (a[:2] + b[:2])
    if a[2] >= CONF:
        return a[:2]
    if b[2] >= CONF:
        return b[:2]
    return None


def _ffill(x, max_gap=5):
    """Causal gap filling: carry the last valid value forward for at most max_gap frames."""
    x = x.copy(); last, gap = np.nan, 0
    for t in range(len(x)):
        if np.isnan(x[t]):
            gap += 1
            if gap <= max_gap:
                x[t] = last
        else:
            last, gap = x[t], 0
    return x


def causal_height(kpts, neck_depth, min_hist=15):
    """Relative neck height above the floor, using only frames <= t. ~1 standing, near 0 on the floor."""
    T = len(kpts)
    ny = np.full(T, np.nan); ay = np.full(T, np.nan)
    for t in range(T):
        p = _mid(kpts[t], L_SH, R_SH)
        if p is None:
            p = _mid(kpts[t], L_HIP, R_HIP)          # hip itself, no upward offset
        if p is None and kpts[t, NOSE, 2] >= CONF:
            p = kpts[t, NOSE, :2]
        if p is not None:
            ny[t] = p[1]
        a = _mid(kpts[t], L_ANK, R_ANK)
        if a is not None:
            ay[t] = a[1]
    ny = _ffill(ny); d = _ffill(np.asarray(neck_depth, dtype=float))
    h = np.full(T, np.nan)
    for t in range(T):
        if np.isnan(ny[t]):
            continue
        dpast = d[:t + 1][~np.isnan(d[:t + 1])]
        scale = np.clip(d[t] / np.median(dpast), 0.5, 2.0) if len(dpast) and not np.isnan(d[t]) and np.median(dpast) > 1e-6 else 1.0
        both = ~np.isnan(ay[:t + 1]) & ~np.isnan(ny[:t + 1])
        if both.sum() >= min_hist:
            span = np.percentile(ay[:t + 1][both] - ny[:t + 1][both], 85)
            floor = np.percentile(ay[:t + 1][both], 90)
        elif not np.isnan(ay[t]):                     # warm-up: the person is assumed upright at the start
            span = ay[t] - ny[t]; floor = ay[t]
        else:
            continue
        span = max(span, 40.0)
        h[t] = np.clip((floor - ny[t]) / (span * scale), -0.2, 1.5)
    return _ffill(h)


def descent_speed(h, lag_s=0.2):
    lag = max(1, int(round(lag_s * FPS)))
    v = np.zeros_like(h)
    v[lag:] = (h[:-lag] - h[lag:]) / (lag / FPS)
    return np.nan_to_num(v)


def trailing_majority(x, win):
    x = x.astype(int); c = np.cumsum(np.concatenate([[0], x])); out = np.zeros(len(x), int)
    for t in range(len(x)):
        lo = max(0, t - win + 1); out[t] = (c[t + 1] - c[lo]) >= (min(win, t + 1) // 2 + 1)
    return out


class SymbolicV2:
    def __init__(self, tau_h=0.35, tau_v=0.45, tau_s=0.5, smooth_s=0.25, lookback_s=1.5):
        self.tau_h, self.tau_v, self.tau_s = tau_h, tau_v, tau_s
        self.smooth, self.lookback = max(1, int(round(smooth_s * FPS))), int(round(lookback_s * FPS))

    def features(self, kpts, neck_depth):
        h = causal_height(kpts, neck_depth)
        return h, descent_speed(h)

    def predict(self, h, v):
        hn = np.nan_to_num(h, nan=1.0)                # unknown height counts as not low
        trigger = trailing_majority((hn < self.tau_h) & (v > self.tau_v), self.smooth)
        low = hn < self.tau_s; fast = v > self.tau_v
        state = np.zeros(len(h), int); on = False
        for t in range(len(h)):
            recent = fast[max(0, t - self.lookback):t + 1].any()
            on = low[t] and (on or recent)
            state[t] = on
        return dict(trigger=trigger, state=state, h=h, v=v)


def support_window(state, win=80):
    c = np.cumsum(np.concatenate([[0], state.astype(int)]))
    return np.array([(c[t + 1] - c[max(0, t - win + 1)]) > 0 for t in range(len(state))], int)


def fuse(neural_prob, sym, win=80, thr=0.5):
    """Paper Eq. 11 to 13: (neural OR trigger) AND symbolic support in the last `win` frames."""
    return (((neural_prob >= thr) | (sym["trigger"] == 1)) & (support_window(sym["state"], win) == 1)).astype(int)


def mcc(y, p):
    tp = int(((y == 1) & (p == 1)).sum()); tn = int(((y == 0) & (p == 0)).sum())
    fp = int(((y == 0) & (p == 1)).sum()); fn = int(((y == 1) & (p == 0)).sum())
    d = np.sqrt(float((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)))
    return (tp * tn - fp * fn) / d if d > 0 else 0.0


def fit(train_feats, lying_h, height_grid=np.linspace(0.10, 0.80, 15), velocity_grid=np.linspace(0.25, 2.0, 15)):
    """train_feats: list of (h, v, labels) for TRAINING videos only; lying_h: heights of the training frames
    URFD labels 'lying on the ground'. tau_h, tau_v: best trigger MCC (paper rule).
    tau_s: 95th percentile of lying_h."""
    Y = np.concatenate([y for _, _, y in train_feats])
    best = (-2, None, None)
    for th in height_grid:
        for tv in velocity_grid:
            m = SymbolicV2(th, tv)
            P = np.concatenate([m.predict(h, v)["trigger"] for h, v, _ in train_feats])
            s = mcc(Y, P)
            if s > best[0]:
                best = (s, th, tv)
    tau_s = float(np.percentile(lying_h, 95))
    return SymbolicV2(best[1], best[2], tau_s), best[0]


## 2. Load our URFD files

In [2]:
PROJECT_ROOT = os.environ.get("PROJECT_ROOT", ".")
NPY = os.path.join(PROJECT_ROOT, "processed_data", "cfg_d31bfed741ab", "urfd_npy")
split = json.load(open(os.path.join(PROJECT_ROOT, "splits", "split_scene_v2.json")))
URFD_FALLS_CSV = os.path.join(PROJECT_ROOT, "URFD", "features", "urfall-cam0-falls.csv")   # for the 'lying' frames (label 1)

feats = {}
for vid in sorted(split["group_of"]):
    k = np.load(os.path.join(NPY, f"{vid}_raw_coco17.npy"))
    nd = np.load(os.path.join(NPY, f"{vid}_neck_depth.npy"))
    y = np.load(os.path.join(NPY, f"{vid}_frame_labels.npy")).astype(int)
    h = causal_height(k, nd)
    feats[vid] = (h, descent_speed(h), y)
cols = ["v", "f", "lab", "hw", "mm", "occ", "std", "hh", "H", "Dmm", "p40"]
urfd = pd.read_csv(URFD_FALLS_CSV, header=None, names=cols)
print(len(feats), "videos")

70 videos


## 3. Fit on train, score validation and test

In [3]:
def lying_heights(vids):
    out = []
    for v in vids:
        if not v.startswith("fall"):
            continue
        h = np.nan_to_num(feats[v][0], nan=1.0)
        i = urfd[(urfd.v == v) & (urfd.lab == 1)].f.values - 1
        out.append(h[i[i < len(h)]])
    return np.concatenate(out)


def report(rows):
    y = np.concatenate([r[1] for r in rows]); p = np.concatenate([r[2] for r in rows])
    tp = int(((y == 1) & (p == 1)).sum()); fp = int(((y == 0) & (p == 1)).sum())
    fn = int(((y == 1) & (p == 0)).sum()); tn = int(((y == 0) & (p == 0)).sum())
    pr, rc = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    det = falls = clean = adl = eps = fr = 0
    for v, yy, pp in rows:        # event definitions of the ST-GCN reports
        if v.startswith("fall"):
            falls += 1; det += int(pp[int(np.argmax(yy)):].any())
        else:
            adl += 1; clean += int(not pp.any()); fr += len(pp); eps += int(pp[0]) + int((np.diff(pp) == 1).sum())
    return dict(MCC=round(mcc(y, p), 3), F1=round(2 * pr * rc / max(pr + rc, 1e-9), 3), Precision=round(pr, 3),
                Recall=round(rc, 3), Specificity=round(tn / max(tn + fp, 1), 3), Accuracy=round((tp + tn) / len(y), 3),
                Falls=f"{det}/{falls}", ADL_no_alarm=f"{clean}/{adl}", Alarms_per_min=round(eps / (fr / FPS / 60), 2))


models, outputs = {}, {}
for f in split["folds"]:
    m, train_mcc = fit([feats[v] for v in f["train"]], lying_heights(f["train"]))
    models[f["fold"]] = m
    print(f"fold {f['fold']}: tau_h={m.tau_h:.3f} tau_v={m.tau_v:.3f} tau_s={m.tau_s:.3f} | train trigger MCC={train_mcc:.3f}")
    for part in ("val", "test"):
        for v in f[part]:
            outputs[(part, v)] = dict(fold=f["fold"], **m.predict(feats[v][0], feats[v][1]))

for part in ("val", "test"):
    for key in ("trigger", "state"):
        rows = [(v, feats[v][2], o[key]) for (p, v), o in outputs.items() if p == part]
        print(f"{part:4s} {key:8s}", report(rows))

fold 0: tau_h=0.550 tau_v=0.375 tau_s=0.359 | train trigger MCC=0.439


fold 1: tau_h=0.750 tau_v=0.375 tau_s=0.130 | train trigger MCC=0.502
val  trigger  {'MCC': np.float64(0.355), 'F1': 0.41, 'Precision': 0.492, 'Recall': 0.352, 'Specificity': 0.954, 'Accuracy': 0.886, 'Falls': '6/6', 'ADL_no_alarm': '1/11', 'Alarms_per_min': 6.84}
val  state    {'MCC': np.float64(0.202), 'F1': 0.306, 'Precision': 0.212, 'Recall': 0.55, 'Specificity': 0.74, 'Accuracy': 0.719, 'Falls': '5/6', 'ADL_no_alarm': '1/11', 'Alarms_per_min': 7.41}
test trigger  {'MCC': np.float64(0.409), 'F1': 0.448, 'Precision': 0.649, 'Recall': 0.342, 'Specificity': 0.967, 'Accuracy': 0.873, 'Falls': '30/30', 'ADL_no_alarm': '7/40', 'Alarms_per_min': 9.26}
test state    {'MCC': np.float64(0.29), 'F1': 0.407, 'Precision': 0.367, 'Recall': 0.455, 'Specificity': 0.86, 'Accuracy': 0.799, 'Falls': '25/30', 'ADL_no_alarm': '20/40', 'Alarms_per_min': 9.06}


## 4. Save the outputs for later use
Per frame, in the same order as `*_frame_labels.npy`: `trigger` (Eq. 8 AND 9, smoothed), `state` (fall state), `support80` (fall state anywhere in the last 80 frames), `h` (relative neck height), `v` (descent speed). Only the test-fold outputs are saved: every URFD video is in exactly one test fold and is scored by the model whose training never saw its room.

In [4]:
save = {}
for (p, v), o in outputs.items():
    if p != "test":
        continue
    save[f"{v}__trigger"] = o["trigger"]; save[f"{v}__state"] = o["state"]
    save[f"{v}__support80"] = support_window(o["state"]); save[f"{v}__h"] = o["h"]; save[f"{v}__v"] = o["v"]
np.savez(os.path.join(PROJECT_ROOT, "symbolic_v2_predictions.npz"), **save)
print("saved symbolic_v2_predictions.npz")

saved symbolic_v2_predictions.npz
